# Sprint 5 Stage 2 — CPU geometry analysis

This notebook consumes a published `geometry-manifest.json` cache only. It never loads a model or raw dataset. Run it after Stage 1 against the Stage 1 cache directory. Set SRC_DIR, V1_GEOMETRY_CACHE, and V1_GEOMETRY_ANALYSIS in the first cell; the exact configured paths are used.


In [ ]:
from pathlib import Path
import json
import os
import sys
import zipfile

CACHE_ROOT = Path(os.environ.get('V1_GEOMETRY_CACHE', '/home/trietlm/anomaly-representation-learning/experiments/20260905/server-geometry-run-1/geometry-cache'))
OUTPUT_DIR = Path(os.environ.get('V1_GEOMETRY_ANALYSIS', '/home/trietlm/anomaly-representation-learning/experiments/20260905/server-geometry-run-1/geometry-analysis'))
NEIGHBOR_K = 15
MAX_NEIGHBOR_QUERIES = 5000
MAX_PROJECTION_SAMPLES = 5000
PROJECTION_SEED = 20260904
NEIGHBOR_SEED = 20260905
TSNE_PERPLEXITY = 30.0

if not (CACHE_ROOT / 'geometry-manifest.json').is_file():
    raise FileNotFoundError(f"Geometry cache manifest not found at {CACHE_ROOT / 'geometry-manifest.json'}. Set V1_GEOMETRY_CACHE to the published Stage 1 cache directory.")


## Cache-only boundary
The analysis stage imports only CPU diagnostics. It validates manifest checksums, numeric array shape, contiguous row linkage, and explicit missing metadata before computing metrics. Labels are used for post-hoc summaries and plot colors only.

In [ ]:
import zipfile

# ---- Server paths: repository source (edit these one-line values; used exactly) ----
# Run notebooks from the repository root, or set V1_REPO_ROOT to the checkout path.
V1_REPO_ROOT = os.environ.get('V1_REPO_ROOT', '/home/trietlm/anomaly-representation-learning')
SRC_DIR = os.environ.get('V1_SRC_DIR', str(Path(V1_REPO_ROOT) / 'src'))

_source_dir = Path(SRC_DIR).expanduser()
if not (_source_dir / 'representation').is_dir():
    raise FileNotFoundError(
        f"Repository source not found: SRC_DIR={_source_dir} has no 'representation' package. "
        f"Run from the repository root or set V1_REPO_ROOT / SRC_DIR to the checkout.")
_source_resolved = str(_source_dir.resolve())
if _source_resolved not in sys.path:
    sys.path.insert(0, _source_resolved)
print(f"[Env] Loaded representation modules from: {_source_dir}")

from representation.diagnostics import analyze_geometry_cache, load_geometry_cache
from representation.geometry_contracts import NeighborConfig, ProjectionConfig

# Fail early with a readable message when Stage 1 was not published or was corrupted.
try:
    _, records, manifest = load_geometry_cache(CACHE_ROOT)
except ValueError as exc:
    raise RuntimeError(f'Geometry cache validation failed: {exc}') from exc
print(f'Validated schema {manifest.schema_version} cache with {len(records)} records')


## Bounded analysis
PCA and t-SNE fit only a seeded bounded sample. Neighbor queries are capped and ties are stable. Figures are static PNGs with counts and legends.

In [ ]:
result = analyze_geometry_cache(
    CACHE_ROOT,
    output_dir=OUTPUT_DIR,
    neighbor_config=NeighborConfig(k=NEIGHBOR_K, max_queries=MAX_NEIGHBOR_QUERIES, seed=NEIGHBOR_SEED),
    projection_config=ProjectionConfig(max_samples=MAX_PROJECTION_SAMPLES, seed=PROJECTION_SEED, perplexity=TSNE_PERPLEXITY),
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(result['metrics'], indent=2, sort_keys=True), encoding='utf-8')
print(json.dumps({'records': result['manifest'].records_count, 'figures': len(result['figures']), 'neighbors': len(result['neighbors'])}, indent=2))


In [ ]:
# Package all cache-linked analysis outputs for download.
download = OUTPUT_DIR / 'geometry-diagnostics.zip'
with zipfile.ZipFile(download, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(CACHE_ROOT.rglob('*')):
        if path.is_file() and path.name != download.name:
            archive.write(path, path.relative_to(CACHE_ROOT).as_posix())
    for path in sorted(OUTPUT_DIR.rglob('*')):
        if path.is_file() and path != download:
            archive.write(path, f'analysis/{path.relative_to(OUTPUT_DIR).as_posix()}')
print(f'Downloadable output: {download}')
